In [1]:
# Importing all the necessary libraries 
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

# Step1: Data Acquisition & Preprocessing 

## 1. Downloading the dataset

In [2]:
# Loading the Breast Cancer dataset from sci-kit learn
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

In [3]:
# Printing the top 5 rows of the dataset
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [4]:
# saving the dataset to a csv file 
df.to_csv('breast_cancer_data.csv', index=False)

## 2. Data Preparation

In [5]:
# Code to get shape of the dataset (row, column)
df.shape

(569, 31)

In [6]:
# Concise summary of the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness error         5

The dataset does not have any null values and the data type of all the columns look suitable meaning that the dataset is cleaned and is ready for analysis.

In [7]:
# Defining the dependent (Y) and independent (X) variables 
X = df.drop(columns=['target'])
y = df['target']

In [8]:
# Splitting the dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
# Scaling the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 2: Feature Selection

In [10]:
from sklearn.feature_selection import SelectKBest, f_classif

def feature_selection(X, y):
    selector = SelectKBest(score_func=f_classif, k='all')
    X_new = selector.fit_transform(X, y)
    return X_new, selector

if __name__ == "__main__":
    X = df.drop(columns=['target'])
    y = df['target']
    X_new, selector = feature_selection(X, y)
    print(X_new.shape)

(569, 30)


In [12]:
# Feature selection to select top 10 features
def feature_selection(X, y):
    selector = SelectKBest(score_func=f_classif, k=10)
    X_new = selector.fit_transform(X, y)
    selected_features = selector.get_support(indices=True)
    return X_new, selected_features

if __name__ == "__main__":
    X = df.drop(columns=['target'])
    y = df['target']
    X_new, selected_features = feature_selection(X, y)
    print(f"Selected features indices: {selected_features}")
    print(f"Selected features names: {X.columns[selected_features].tolist()}")
    print(X_new.shape)

Selected features indices: [ 0  2  3  6  7 20 22 23 26 27]
Selected features names: ['mean radius', 'mean perimeter', 'mean area', 'mean concavity', 'mean concave points', 'worst radius', 'worst perimeter', 'worst area', 'worst concavity', 'worst concave points']
(569, 10)


# Step 3: Grid Search CV for Model Tuning  

In [12]:
# Grid Search Cross-Validation to find the best model
def grid_search_cv(X, y):
    parameters = {
        'hidden_layer_sizes': [(50, 50, 50), (50, 100, 50), (100,)],
        'activation': ['tanh', 'relu'],
        'solver': ['sgd', 'adam'],
        'alpha': [0.0001, 0.05],
        'learning_rate': ['constant', 'adaptive'],
    }
    mlp = MLPClassifier(max_iter=100)
    clf = GridSearchCV(mlp, parameters, n_jobs=-1, cv=5)
    clf.fit(X, y)
    return clf

# Model Evaluation

In [21]:
if __name__ == "__main__":
    # Loading and preprocessing the dataset

    X = df.drop(columns=['target'])
    y = df['target']

    # Performing feature selection
    X_new, selected_features = feature_selection(X, y)
    selected_feature_names = X.columns[selected_features].tolist()
    print(f"Selected feature names: {selected_feature_names}")

    # Performing Grid Search CV to find the best model
    clf = grid_search_cv(X_new, y)
    print(f"Best parameters found: {clf.best_params_}")

    # Training the best model
    best_model = clf.best_estimator_
    best_model.fit(X_new, y)

    # Evaluate the best model
    evaluate_model(best_model, X_new, y)

Selected feature names: ['mean radius', 'mean perimeter', 'mean area', 'mean concavity', 'mean concave points', 'worst radius', 'worst perimeter', 'worst area', 'worst concavity', 'worst concave points']
Best parameters found: {'activation': 'tanh', 'alpha': 0.0001, 'hidden_layer_sizes': (50, 50, 50), 'learning_rate': 'adaptive', 'solver': 'adam'}
              precision    recall  f1-score   support

           0       0.92      0.82      0.87       212
           1       0.90      0.96      0.93       357

    accuracy                           0.91       569
   macro avg       0.91      0.89      0.90       569
weighted avg       0.91      0.91      0.91       569



In [22]:
# Save the model, scaler, and selector
joblib.dump(best_model, 'final_model.pkl')
joblib.dump(selector, 'selector.pkl')

['selector.pkl']